# Flight Delay Analysis
**Dataset:** 2015 US domestic flights (~5.8M rows)  
**Goal:** Predict arrival/departure delays

## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

DATA_DIR = Path('..')

## 2. Data Loading

In [ ]:
# Explicit dtypes cut memory usage roughly in half
FLIGHT_DTYPES = {
   # 'YEAR': 'int16', # alles 2015 -> raus
    
    'MONTH': 'int8', 
    'DAY': 'int8', 
    'DAY_OF_WEEK': 'int8',

    'FLIGHT_NUMBER': 'int32', 
    
    #'TAIL_NUMBER' viele missing 
    'SCHEDULED_DEPARTURE': 'string',  
    'DEPARTURE_TIME': 'string', #1% missing -> raus
    'DEPARTURE_DELAY': 'float32', #1% missing = DEPARTURE_TIME


    #'TAXI_OUT': 'float32', # 2% viele missing -> raus
    #'WHEELS_OFF': 'string', # 2% viele missing -> raus = TAX_OFF

    'SCHEDULED_TIME': 'float32', # 6 mising-> raus
    'ELAPSED_TIME': 'float32',# 2% viele missing -> raus
    #'AIR_TIME': 'float32',# 2% viele missing -> raus =ELAPSED
    'DISTANCE': 'int32',
    
    
    #'WHEELS_ON': 'string',  # # 2% viele missing -> raus
    #'TAXI_ON': 'float32', # 2% viele missing -> raus
    'SCHEDULED_ARRIVAL' : 'string', 
    'ARRIVAL_TIME': 'string',
    'ARRIVAL_DELAY': 'float32', 

    
    #'DIVERTED': 'bool', # raus -> es gibt nur 0
    #'CANCELLED': 'bool',  # 99.99% sind 0 -> raus

    #'AIRSYSTEM_DELAY'  _> raus
    #'SECURITY_DELAY'  _> raus
    #'AIRLINE_DELAY'  _> raus
    #'LATEAIRCRAFT_DELAY'  _> raus
    #'WEATHER_DELAY'  _> raus

}

print('Loading flights.csv (~592 MB) ...')
flights = pd.read_csv(DATA_DIR / 'flights.csv', dtype=FLIGHT_DTYPES)
airlines = pd.read_csv(DATA_DIR / 'airlines.csv')
airports = pd.read_csv(DATA_DIR / 'airports.csv')

# Keep only columns defined in FLIGHT_DTYPES
flights = flights[list(FLIGHT_DTYPES.keys())]
before = print(len(flights))

# Drop rows with any missing value

flights = flights.dropna()

print(len(flights))

print(f'flights: {flights.shape[0]:,} rows × {flights.shape[1]} cols')
print(f'airlines: {airlines.shape[0]} | airports: {airports.shape[0]}')

Loading flights.csv (~592 MB) ...


/tmp/ipykernel_9834/1703299786.py:45: DtypeWarning: Columns (0: ORIGIN_AIRPORT, 1: DESTINATION_AIRPORT) have mixed types. Specify dtype option on import or set low_memory=False.
  flights = pd.read_csv(DATA_DIR / 'flights.csv', dtype=FLIGHT_DTYPES)


5819079
5714008
flights: 5,714,008 rows × 13 cols
airlines: 14 | airports: 322


In [28]:
out_path = DATA_DIR / 'flights_cut.csv'
flights.to_csv(out_path, index=False)
print(f'Saved {flights.shape[0]:,} rows × {flights.shape[1]} cols → {out_path}')

Saved 100 rows × 16 cols → ../flights_cut.csv


In [15]:
flights.head(10)

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,0005,...,0408,-22.0,False,False,NaN,NaN,NaN,NaN,NaN,NaN
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,0010,...,0741,-9.0,False,False,NaN,NaN,NaN,NaN,NaN,NaN
2,2015,1,1,4,US,840,N171US,SFO,CLT,0020,...,0811,5.0,False,False,NaN,NaN,NaN,NaN,NaN,NaN
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,0020,...,0756,-9.0,False,False,NaN,NaN,NaN,NaN,NaN,NaN
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,0025,...,0259,-21.0,False,False,NaN,NaN,NaN,NaN,NaN,NaN
5,2015,1,1,4,DL,806,N3730B,SFO,MSP,0025,...,0610,8.0,False,False,NaN,NaN,NaN,NaN,NaN,NaN
6,2015,1,1,4,NK,612,N635NK,LAS,MSP,0025,...,0509,-17.0,False,False,NaN,NaN,NaN,NaN,NaN,NaN
7,2015,1,1,4,US,2013,N584UW,LAX,CLT,0030,...,0753,-10.0,False,False,NaN,NaN,NaN,NaN,NaN,NaN
8,2015,1,1,4,AA,1112,N3LAAA,SFO,DFW,0030,...,0532,-13.0,False,False,NaN,NaN,NaN,NaN,NaN,NaN
9,2015,1,1,4,DL,1173,N826DN,LAS,ATL,0030,...,0656,-15.0,False,False,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Exploratory Data Analysis

In [ ]:
# Null overview
null_pct = flights.isnull().mean().mul(100).sort_values(ascending=False)
print('Columns with missing values:')
print(null_pct[null_pct > 0].to_string())

In [ ]:
# Arrival delay distribution (non-cancelled flights)
on_time = flights[flights['CANCELLED'] == 0].copy()
print(f'Non-cancelled flights: {len(on_time):,}')
print(f'Delayed >15 min: {(on_time["ARRIVAL_DELAY"] > 15).mean():.1%}')
print(f'Cancelled: {flights["CANCELLED"].mean():.1%}')

on_time['ARRIVAL_DELAY'].clip(-30, 120).hist(bins=60, edgecolor='none')
plt.axvline(15, color='red', linestyle='--', label='15 min threshold')
plt.xlabel('Arrival Delay (min)')
plt.ylabel('Flights')
plt.title('Arrival Delay Distribution')
plt.legend()
plt.tight_layout()

In [ ]:
# Average delay by month
monthly = on_time.groupby('MONTH')['ARRIVAL_DELAY'].mean()
monthly.plot(kind='bar', color='steelblue')
plt.xlabel('Month')
plt.ylabel('Mean Arrival Delay (min)')
plt.title('Average Delay by Month')
plt.xticks(rotation=0)
plt.tight_layout()

In [ ]:
# Delay by airline
airline_delay = (
    on_time.merge(airlines, left_on='AIRLINE', right_on='IATA_CODE')
    .groupby('AIRLINE_y')['ARRIVAL_DELAY']
    .mean()
    .sort_values()
)
airline_delay.plot(kind='barh', color='steelblue')
plt.xlabel('Mean Arrival Delay (min)')
plt.title('Average Delay by Airline')
plt.tight_layout()

## 4. Feature Engineering

In [ ]:
df = on_time.dropna(subset=['ARRIVAL_DELAY']).copy()

# Cyclical time encoding
df['HOUR'] = df['SCHEDULED_DEPARTURE'] // 100
df['HOUR_SIN'] = np.sin(2 * np.pi * df['HOUR'] / 24)
df['HOUR_COS'] = np.cos(2 * np.pi * df['HOUR'] / 24)
df['MONTH_SIN'] = np.sin(2 * np.pi * df['MONTH'] / 12)
df['MONTH_COS'] = np.cos(2 * np.pi * df['MONTH'] / 12)
df['DOW_SIN'] = np.sin(2 * np.pi * df['DAY_OF_WEEK'] / 7)
df['DOW_COS'] = np.cos(2 * np.pi * df['DAY_OF_WEEK'] / 7)

# Encode categoricals
for col in ['AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']:
    df[col] = df[col].astype('category').cat.codes

# Binary target
df['LATE'] = (df['ARRIVAL_DELAY'] > 15).astype(int)

# Feature columns (no leakage)
LEAKAGE = [
    'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY',
    'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY', 'DEPARTURE_TIME',
    'WHEELS_OFF', 'WHEELS_ON', 'TAXI_IN', 'TAXI_OUT',
    'ELAPSED_TIME', 'AIR_TIME', 'ARRIVAL_TIME', 'DEPARTURE_DELAY',
    'TAIL_NUMBER', 'CANCELLATION_REASON',
]
FEATURES = [c for c in df.columns if c not in LEAKAGE + ['ARRIVAL_DELAY', 'LATE', 'CANCELLED', 'DIVERTED']]
print('Features:', FEATURES)

## 5. Modelling

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

X = df[FEATURES].fillna(0)
y = df['LATE']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=df['MONTH']
)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

In [ ]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
print('Baseline:', classification_report(y_test, baseline.predict(X_test), digits=3))

In [ ]:
model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
model.fit(X_train, y_train)
print(classification_report(y_test, model.predict(X_test), digits=3))

## 6. Feature Importance

In [ ]:
importance = pd.Series(model.feature_importances_, index=FEATURES).nlargest(15)
importance.sort_values().plot(kind='barh', color='steelblue')
plt.xlabel('Importance')
plt.title('Top 15 Feature Importances')
plt.tight_layout()